# Extra Credit: Spark vs Ray Framework Comparison
Beehive Sounds Dataset — SBCM

**Task:** Compute statistics (mean, std, percentiles) on the CSV dataset using both Spark and Ray, then compare performance.

In [1]:
import sys
sys.path.append('/home/stiwari6/.local/lib/python3.11/site-packages')

import kagglehub
import os
import time
import tracemalloc

os.environ["KAGGLE_USERNAME"] = "snigdhatiwarisd"
os.environ["KAGGLE_KEY"] = "KGAT_b5f7b91d08ee5010cc63d7ee3b3e3170"

path = kagglehub.dataset_download("annajyang/beehive-sounds")
csv_path = os.path.join(path, "all_data_updated.csv")
print("csv_path:", csv_path)

csv_path: /home/stiwari6/.cache/kagglehub/datasets/annajyang/beehive-sounds/versions/3/all_data_updated.csv


In [2]:
CONTINOUS_COLS = [
    "hive temp", "hive humidity", "hive pressure",
    "weather temp", "weather humidity", "weather pressure",
    "wind speed", "gust speed", "cloud coverage",
    "rain", "lat", "long", "frames", "time"
]

## Spark Implementation

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("BeehiveStats_Spark") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

df = spark.read.csv(csv_path, header=True, inferSchema=True)
df.cache()
df.count()  # materialize cache
print(f"loaded {df.count()} rows")

loaded 1275 rows


In [4]:
def run_spark_stats():
    # mean, std, min, max, count
    df.select([F.col(f"`{c}`") for c in CONTINOUS_COLS]).describe().collect()

    # percentiles (Q1, median, Q3)
    quantile_aggs = []
    for c in CONTINOUS_COLS:
        quantile_aggs += [
            F.expr(f"percentile_approx(`{c}`, 0.25)").alias(f"{c}_Q1"),
            F.expr(f"percentile_approx(`{c}`, 0.50)").alias(f"{c}_median"),
            F.expr(f"percentile_approx(`{c}`, 0.75)").alias(f"{c}_Q3"),
        ]
    df.agg(*quantile_aggs).collect()

spark_times = []
for i in range(3):
    start = time.time()
    run_spark_stats()
    elapsed = time.time() - start
    spark_times.append(elapsed)
    print(f"  run {i+1}: {elapsed:.2f}s")

spark_avg = sum(spark_times) / len(spark_times)
print(f"\nSpark average: {spark_avg:.2f}s")

  run 1: 1.58s
  run 2: 0.67s
  run 3: 0.57s

Spark average: 0.94s


In [5]:
# peak memory for spark
tracemalloc.start()
run_spark_stats()
_, spark_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
spark_peak_mb = spark_peak_bytes / 1024 / 1024
print(f"Spark peak memory: {spark_peak_mb:.1f} MB")

Spark peak memory: 0.2 MB


In [6]:
# show the actual results as formatted tables
import pandas as pd

# ── descriptive stats ────────────────────────────────────────
desc_pd = (
    df.select([F.col(f"`{c}`") for c in CONTINOUS_COLS])
      .describe()
      .toPandas()
      .set_index("summary")
      .T
      .rename_axis("column")
      .reset_index()
)
for col in desc_pd.columns[1:]:
    desc_pd[col] = pd.to_numeric(desc_pd[col], errors="coerce").round(3)

print("=== Spark Descriptive Statistics ===")
print(desc_pd.to_string(index=False))

# ── percentiles ───────────────────────────────────────────────
quantile_aggs = []
for c in CONTINOUS_COLS:
    quantile_aggs += [
        F.expr(f"percentile_approx(`{c}`, 0.25)").alias(f"{c}_Q1"),
        F.expr(f"percentile_approx(`{c}`, 0.50)").alias(f"{c}_median"),
        F.expr(f"percentile_approx(`{c}`, 0.75)").alias(f"{c}_Q3"),
    ]

q_pd = df.agg(*quantile_aggs).toPandas()

rows = []
for c in CONTINOUS_COLS:
    rows.append({
        "column":  c,
        "Q1":      round(q_pd[f"{c}_Q1"].iloc[0], 3),
        "median":  round(q_pd[f"{c}_median"].iloc[0], 3),
        "Q3":      round(q_pd[f"{c}_Q3"].iloc[0], 3),
    })
q_df = pd.DataFrame(rows)

print("\n=== Spark Percentiles ===")
print(q_df.to_string(index=False))

=== Spark Descriptive Statistics ===
          column  count     mean  stddev     min      max
       hive temp   1275   29.009   8.172   15.50   55.620
   hive humidity   1275   44.664  18.360    7.23   93.470
   hive pressure   1275 1009.179   2.406 1003.54 1015.970
    weather temp   1271   20.327   5.588   10.75   35.430
weather humidity   1275   63.501  16.207    0.00   88.000
weather pressure   1275 1011.370  56.808    0.00 1021.000
      wind speed   1271    3.805   2.311    0.00   10.800
      gust speed    281    4.511   3.932    0.45   15.430
  cloud coverage   1275   27.868  33.641    0.00  100.000
            rain   1275    0.000   0.000    0.00    0.000
             lat   1271   37.290   0.000   37.29   37.290
            long   1271 -121.950   0.000 -121.95 -121.950
          frames   1275    9.109   0.994    8.00   10.000
            time   1275    0.483   0.287    0.00    0.958

=== Spark Percentiles ===
          column      Q1  median      Q3
       hive temp   22.45 

## Ray Implementation

In [7]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "ray[data]", "--quiet"], check=True)

CompletedProcess(args=['/opt/conda/bin/python', '-m', 'pip', 'install', 'ray[data]', '--quiet'], returncode=0)

In [8]:
import ray
import ray.data
import numpy as np

ray.init(ignore_reinit_error=True)
print(ray.available_resources())

2026-06-01 17:25:12,713	INFO worker.py:2012 -- Started a local Ray instance.


{'object_store_memory': 64835611852.0, 'memory': 151283094324.0, 'node:198.202.101.91': 1.0, 'node:__internal_head__': 1.0, 'CPU': 128.0}


/home/stiwari6/.local/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [ ]:
def run_ray_stats():
    ds = ray.data.read_csv(csv_path)

    # mean, std, min, max per column
    stats = {}
    for col in CONTINOUS_COLS:
        stats[col] = {
            "mean":  ds.mean(col),
            "std":   ds.std(col),
            "min":   ds.min(col),
            "max":   ds.max(col),
            "count": ds.count(),
        }

    # percentiles via pandas conversion on the numeric subset
    pdf = ds.select_columns(CONTINOUS_COLS).to_pandas()
    quantiles = pdf.quantile([0.25, 0.50, 0.75])

    return stats, quantiles

ray_times = []
for i in range(3):
    start = time.time()
    run_ray_stats()
    elapsed = time.time() - start
    ray_times.append(elapsed)
    print(f"  run {i+1}: {elapsed:.2f}s")

ray_avg = sum(ray_times) / len(ray_times)
print(f"\nRay average: {ray_avg:.2f}s")

2026-06-01 17:25:35,114	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-06-01 17:25:35,364	INFO logging.py:416 -- Registered dataset logger for dataset dataset_2_0
2026-06-01 17:25:35,910	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.2MiB, output=0.2MiB, total=0.3MiB, 
2026-06-01 17:25:35,914	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_2_0. Full logs are in /scratch/stiwari6/job_49960783/ray/session_2026-06-01_17-25-02_623243_22538/logs/ray-data
2026-06-01 17:25:35,915	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]
2026-06-01 17:25:35,931	WARNING resource_manager.py:169 -- ⚠️  Ray's object s

In [14]:
# peak memory for ray
tracemalloc.start()
run_ray_stats()
_, ray_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
ray_peak_mb = ray_peak_bytes / 1024 / 1024
print(f"Ray peak memory: {ray_peak_mb:.1f} MB")

2026-05-29 10:39:57,644	INFO logging.py:416 -- Registered dataset logger for dataset dataset_386_0
2026-05-29 10:39:57,649	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.2MiB, output=0.2MiB, total=0.3MiB, 
2026-05-29 10:39:57,655	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_386_0. Full logs are in /scratch/stiwari6/job_49871179/ray/session_2026-05-29_10-26-19_332726_1065482/logs/ray-data
2026-05-29 10:39:57,656	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_386_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]
2026-05-29 10:39:57,700	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_386_0 =======
2026-05-29 10:39:57,701	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-29 10:39:57,703	INFO logging_progr

Ray peak memory: 20.3 MB


In [15]:
# show actual results
ray_stats, ray_quantiles = run_ray_stats()

print("=== Ray Descriptive Statistics ===")
import pandas as pd
rows = []
for col, s in ray_stats.items():
    rows.append({"column": col, "count": s["count"], "mean": round(s["mean"],4),
                 "std": round(s["std"],4), "min": round(s["min"],4), "max": round(s["max"],4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Ray Percentiles ===")
print(ray_quantiles.to_string())

2026-05-29 10:47:39,076	INFO logging.py:416 -- Registered dataset logger for dataset dataset_514_0
2026-05-29 10:47:39,080	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.2MiB, output=0.2MiB, total=0.3MiB, 
2026-05-29 10:47:39,083	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_514_0. Full logs are in /scratch/stiwari6/job_49871179/ray/session_2026-05-29_10-26-19_332726_1065482/logs/ray-data
2026-05-29 10:47:39,084	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_514_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]
2026-05-29 10:47:39,103	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_514_0 =======
2026-05-29 10:47:39,104	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-29 10:47:39,104	INFO logging_progr

=== Ray Descriptive Statistics ===
          column  count      mean     std     min      max
       hive temp   1275   29.0095  8.1724   15.50   55.620
   hive humidity   1275   44.6639 18.3597    7.23   93.470
   hive pressure   1275 1009.1792  2.4060 1003.54 1015.970
    weather temp   1275   20.3268  5.5879   10.75   35.430
weather humidity   1275   63.5012 16.2072    0.00   88.000
weather pressure   1275 1011.3702 56.8079    0.00 1021.000
      wind speed   1275    3.8055  2.3112    0.00   10.800
      gust speed   1275    4.5113  3.9318    0.45   15.430
  cloud coverage   1275   27.8682 33.6413    0.00  100.000
            rain   1275    0.0000  0.0000    0.00    0.000
             lat   1275   37.2900  0.0000   37.29   37.290
            long   1275 -121.9500  0.0000 -121.95 -121.950
          frames   1275    9.1090  0.9944    8.00   10.000
            time   1275    0.4830  0.2874    0.00    0.958

=== Ray Percentiles ===
      hive temp  hive humidity  hive pressure  weather 

## Performance Comparison

In [16]:
import pandas as pd

spark_loc = """df.select(...).describe().collect()
df.agg(*quantile_aggs).collect()""".strip().count('\n') + 2 + len(CONTINOUS_COLS)*3 + 5

ray_loc = """ds = ray.data.read_csv(csv_path)
for col in CONTINOUS_COLS: ds.mean/std/min/max/count
pdf = ds.select_columns(...).to_pandas()
quantiles = pdf.quantile([0.25,0.50,0.75])""".strip().count('\n') + 4

comparison = pd.DataFrame({
    "Metric":         ["Avg execution time (3 runs)", "Lines of code", "Peak memory (MB)"],
    "Spark":          [f"{spark_avg:.2f}s", "~15", f"{spark_peak_mb:.1f}"],
    "Ray":            [f"{ray_avg:.2f}s",   "~20", f"{ray_peak_mb:.1f}"],
})
print(comparison.to_string(index=False))

                     Metric Spark     Ray
Avg execution time (3 runs) 1.01s 142.94s
              Lines of code   ~15     ~20
           Peak memory (MB)   0.2    20.3


## Analysis

**Which framework was faster? By how much?**
Spark was faster for this task. The beehive CSV sits on Expanse's Lustre filesystem, and `describe()` + `percentile_approx` are single-pass aggregations that Spark handles efficiently in one call. Ray reads the CSV sequentially before distributing the work, so the overhead ends up outweighing any parallelism benefit at this dataset size. Spark averaged 1.01s vs Ray's 142.94s.

**Which was easier to implement? Why?**
Spark was easier to implement. `df.describe()` and `percentile_approx` produce a clean summary in one call. Ray Data doesn't have a built-in percentile aggregator, so we had to convert to pandas to compute quantiles, which added an extra step and made the code more complicated.

**For your specific use case, which would you choose?**
We would choose Spark for tabular statistics and preprocessing. Ray would be the better choice if the pipeline included the audio feature extraction UDF at scale or model training with PyTorch, since Ray's integrations (Ray Train, Ray Tune) give it a clear advantage over Spark's MLlib for those kinds of tasks.